[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C40_Research_Methodology_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy / pandas、CPU 可跑**。这门课偏方法论，但**不是只讲不练**——每条研究原则都配一个能跑、能 `assert` 的小实验。

这个 notebook 做三件事：① 确认环境；② 用一个 60 秒的小实验，让你**亲眼看到**「只跑一个随机种子」如何骗了你（本课最重要的直觉）；③ 立下全课的纪律——**研究 = 主张 + 证据，且证据要配得上主张**。

## 1 · 环境自检

只需要 `numpy` 与 `pandas`。`matplotlib` 可选（仅用于画图）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
import pandas as pd
print('numpy', np.__version__)
print('pandas', pd.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 当头一棒：只跑一个种子会骗你

假设你提出方法 B，想证明它优于基线 A。你各跑**一次**，B 赢了，于是宣布「B 更好」。

下面我们**作弊地**让 A 和 B 其实**一模一样好**（真实差异 = 0），只是每次运行有随机波动（随机种子方差）。看看「只跑一次」会得出什么结论。

In [ ]:
rng = np.random.default_rng(0)

# 真相：A 和 B 的『真实』准确率完全相同 = 0.80，但每次运行有 ±0.03 的随机种子波动
TRUE_ACC = 0.80
NOISE = 0.03

def run_once(rng):
    a = TRUE_ACC + rng.normal(0, NOISE)   # 方法 A 的一次运行
    b = TRUE_ACC + rng.normal(0, NOISE)   # 方法 B 的一次运行（真实水平与 A 相同！）
    return a, b

# 模拟 1000 个研究者，每人只跑一次 A、一次 B，然后宣布赢家
b_wins = 0
for _ in range(1000):
    a, b = run_once(rng)
    if b > a:
        b_wins += 1
print(f'真相：A 与 B 真实水平完全相同（差异=0）')
print(f'但只跑一次时，B「赢」的比例 = {b_wins/1000:.0%}')
assert 0.4 < b_wins/1000 < 0.6, '应约一半时间 B 赢、一半 A 赢——纯属运气'
print('\n教训：差异为 0 时，单次运行约一半概率「赢」。一次实验的胜负几乎是抛硬币。')

**关键结论**：当真实差异很小（相对噪声）时，单次运行的胜负基本是**运气**。

你随手宣布的「B 更好」，换个种子可能就反过来。这就是为什么本课模块 03 反复强调：**报多次运行、报方差、做统计检验**。一个数字没有方差，等于没有信息。

## 3 · 多跑几次：让信号浮出噪声

现在给 B 一个**真实**的小优势（+0.02），但噪声依旧。比较「只跑一次的胜负」与「跑 N 次取均值」谁更能识别真实差异。

In [ ]:
def trial(rng, n_runs, true_gap=0.02):
    '''A 真实=0.80，B 真实=0.80+true_gap。各跑 n_runs 次取均值，返回 B 均值是否 > A 均值。'''
    a_runs = 0.80 + rng.normal(0, NOISE, size=n_runs)
    b_runs = 0.80 + true_gap + rng.normal(0, NOISE, size=n_runs)
    return b_runs.mean() > a_runs.mean()

print(f"{'每组跑几次':>10s} {'B 正确判赢的比例':>18s}")
results = {}
for n_runs in [1, 3, 10, 30]:
    wins = sum(trial(rng, n_runs) for _ in range(2000)) / 2000
    results[n_runs] = wins
    print(f'{n_runs:>10d} {wins:>17.0%}')
# 跑得越多，越能稳定识别 B 的真实优势（胜率从~接近抛硬币 升到 接近 1）
assert results[1] < results[30], '跑得越多应越能识别真实差异'
assert results[30] > 0.8, '跑 30 次应能可靠识别 +0.02 的真实优势'
print('\n✅ 同一个真实的 +0.02 优势：跑 1 次几乎看不出，跑 30 次几乎必然识别。')
print('这就是「样本量 / 种子数」的威力 —— 模块 03 会量化「该跑多少次」。')

## 4 · 立纪律一：研究 = 主张 + 证据，证据要配得上主张

本课的核心世界观：每个**主张**都需要**证据**支撑，而且**主张越大，需要的证据越强**。

我们把这条原则做成一个可计算的小工具：给定主张的「大小」与证据的「强度」，判断证据是否**撑得起**主张（即有没有过度声明）。

In [ ]:
def claim_supported(claim_strength, evidence_strength):
    '''主张越强，要求证据越强。证据 >= 主张 才算「撑得起」（非过度声明）。
       claim_strength / evidence_strength: 1~5 的等级。'''
    return evidence_strength >= claim_strength

# 例子：弱证据（1次运行无对照=1）配强主张（颠覆性结论=5）-> 过度声明
cases = [
    ('一次运行无对照',       1, '本方法颠覆该领域',     5),
    ('多种子+对照+消融+显著', 4, '本方法颠覆该领域',     5),
    ('多种子+对照+消融+显著', 4, '本方法在任务T小幅优于B', 2),
]
print(f"{'证据':<22}{'证据强':>5}{'主张':<22}{'主张强':>5}{'撑得起?':>8}")
for ev_name, ev, cl_name, cl in cases:
    ok = claim_supported(cl, ev)
    print(f'{ev_name:<22}{ev:>5}{cl_name:<22}{cl:>5}{("是 ✅" if ok else "否 ❌过度声明"):>10}')
assert not claim_supported(5, 1), '弱证据配强主张=过度声明'
assert claim_supported(2, 4), '强证据配弱主张=稳妥'
print('\n✅ 这就是读论文与写论文时反复要做的『称重』：证据撑得起主张吗？')

## 5 · 立纪律二：用 pandas 聚合多次运行（贯穿全课的工具）

本课大量使用 pandas 把「多次运行」聚合成「均值 ± 标准差」的可读表。先把这个工作流跑通——后面每个模块都用它。

In [ ]:
# 模拟两个方法各跑 5 个种子的准确率
rng = np.random.default_rng(42)
records = []
for method, true_acc in [('baseline_A', 0.80), ('method_B', 0.82)]:
    for seed in range(5):
        acc = true_acc + rng.normal(0, 0.03)
        records.append(dict(method=method, seed=seed, accuracy=acc))
df = pd.DataFrame(records)
print('原始记录（长表）：')
print(df)

# 按方法聚合：均值、标准差、运行次数
summary = df.groupby('method')['accuracy'].agg(['mean', 'std', 'count']).round(4)
print('\n聚合后（每个方法的 均值±标准差）：')
print(summary)
assert set(summary.index) == {'baseline_A', 'method_B'}
assert (summary['count'] == 5).all(), '每个方法应有 5 次运行'
print('\n✅ groupby+agg 是本课聚合实验结果的主力工具。注意：报均值必须同时报 std（或 CI）。')

## 6 · 一个会贯穿全课的「证据强度检查表」

把「一个结果可不可信」拆成可勾选的若干项，封装成函数。后面每个模块都会回到这张检查表。

In [ ]:
def credibility_score(has_control, has_ablation, multi_seed, reports_variance,
                      strong_baseline, not_overclaimed, reproducible):
    '''把一个结果的可信度拆成 7 项布尔检查，返回 (得分0~7, 缺失项列表)。'''
    checks = {
        '有对照组':        has_control,
        '有消融归因':      has_ablation,
        '多种子运行':      multi_seed,
        '报告了方差':      reports_variance,
        '基线够强':        strong_baseline,
        '没有过度声明':    not_overclaimed,
        '可复现':          reproducible,
    }
    score = sum(checks.values())
    missing = [k for k, v in checks.items() if not v]
    return score, missing

# 一个『SOTA 但不严谨』的典型结果
score, missing = credibility_score(
    has_control=False, has_ablation=False, multi_seed=False, reports_variance=False,
    strong_baseline=True, not_overclaimed=False, reproducible=True)
print(f'可信度得分 = {score}/7')
print(f'缺失项：{missing}')
assert score == 2, '该例应只满足 2 项'
assert '有对照组' in missing and '报告了方差' in missing
print('\n✅ 这张 7 项检查表是本课的『北极星』：读论文给它打分、做研究照它自查。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：方法论不空谈。你会亲手跑「只跑一个种子如何骗你」「一次改两个量如何让结论作废」「没有误差棒的图如何掩盖真相」，每个都用 `assert` 兜底。

**接下来六个模块**：01 读论文与复现 → 02 实验设计与消融 → 03 研究统计 → 04 研究工程 → 05 写作与审稿。这是研究的完整闭环：读入 → 生产 → 输出。

下一站：**模块 01 · 读论文与复现**。